# P1-E10.2 — Frozen CAUEEG Normal-vs-AD External Validation on AHEPA

## Goal

Perform a single participant-level external evaluation of the **frozen CAUEEG Normal-vs-AD conventional benchmark** on eligible AHEPA participants. The notebook reuses the locked preprocessing, automated epoch QA, feature schema, feature subset, fitted estimator and prediction rule without retraining or outcome-guided modification.

The expected final status is:

> **P1-E10.2 PASSED — AHEPA FROZEN NORMAL-vs-AD EXTERNAL VALIDATION LOCKED**

Passing means the protocol executed correctly; it does **not** require high predictive performance.

## Context & Methods

### Prespecified analysis

- External cohort: AHEPA/OpenNeuro ds004504, CN and AD only.
- Unit of analysis: one BIDS participant.
- FTD is excluded because it is not a CAUEEG target class.
- Preprocessing: frozen reference path, no second CAR, 1–40 Hz Butterworth filter, 200 Hz, 2-second edge trim.
- Epochs: non-overlapping 10-second epochs; automated QA thresholds inherited from P1-E07.1; maximum 12 evenly distributed accepted epochs; minimum 6.
- Representation: exact P1-E08.1 feature equations and aggregation.
- Model: stored P1-E08.1 `normal_vs_ad` artifact, already fitted on CAUEEG development data.
- Prediction: estimator `.predict()` unchanged. No recalibration, threshold adjustment, tuning, class balancing or refitting.
- Labels remain outside preprocessing and feature extraction and are joined only before evaluation.

## Setup

### 1. Paths and frozen constants

In [ ]:
from pathlib import Path
from fractions import Fraction
import hashlib, json, platform, re, sys, traceback, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, brier_score_loss, roc_curve, precision_recall_curve)
import joblib

try:
    import mne
except Exception as exc:
    raise ImportError('Install MNE in eeg_snn: conda install -c conda-forge mne -y') from exc

PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
AHEPA_ROOT=Path(r'D:\EEG_AD_Research\data\ds004504')
E10_1_ROOT=PAPER_ROOT/'p1_e10_1_dual_external_compatibility_protocol_lock'
E08_ROOT=PAPER_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'
OUTPUT_ROOT=PAPER_ROOT/'p1_e10_2_ahepa_frozen_normal_ad_external_validation'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; ARRAYS=OUTPUT_ROOT/'epoch_arrays'
for path in (OUTPUT_ROOT,TABLES,FIGURES,ARRAYS): path.mkdir(parents=True,exist_ok=True)

E10_1_RUN=E10_1_ROOT/'run_record.json'
E10_1_PROTOCOL=E10_1_ROOT/'p1_e10_1_LOCKED_EXTERNAL_VALIDATION_PROTOCOL.json'
AHEPA_MANIFEST=E10_1_ROOT/'tables'/'04_AHEPA_participant_eligibility_manifest.csv'
MODEL_PATH=E08_ROOT/'models'/'normal_vs_ad_LOCKED_NONSPIKING_BENCHMARK.joblib'

SEED=20260828; FS=200.0; BAND=(1.0,40.0); FILTER_ORDER=4
EDGE_TRIM_SECONDS=2.0; EPOCH_SECONDS=10.0; TARGET_EPOCHS=12; MIN_EPOCHS=6
PTP_LIMIT_UV=600.0; RMS_LIMIT_UV=120.0; EXTREME_UV=150.0
MAX_EXTREME_FRACTION=0.02; ROBUST_SCORE_LIMIT=5.0; FLAT_SD_UV=1e-6
MIN_PARTICIPANT_RETENTION=0.90; BOOTSTRAP_REPS=2000
CHANNELS=['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'low_gamma':(30,40)}

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9,'axes.titlesize':10,
                     'axes.labelsize':9,'pdf.fonttype':42,'ps.fonttype':42})
sns.set_theme(style='whitegrid',context='paper'); mne.set_log_level('ERROR')

def sha256_file(path,block_size=4*1024*1024):
    d=hashlib.sha256()
    with Path(path).open('rb') as h:
        for block in iter(lambda:h.read(block_size),b''): d.update(block)
    return d.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name,width=8,height=4):
    fig.set_size_inches(width,height); fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight',facecolor='white')
def robust_z(values):
    values=np.asarray(values,float); med=np.nanmedian(values); mad=np.nanmedian(np.abs(values-med))
    return np.zeros_like(values) if not np.isfinite(mad) or mad<1e-12 else (values-med)/(1.4826*mad)

print('Experiment: P1-E10.2'); print('AHEPA:',AHEPA_ROOT); print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate P1-E10.1 and the frozen CAUEEG model

In [ ]:
required=[E10_1_RUN,E10_1_PROTOCOL,AHEPA_MANIFEST,MODEL_PATH]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('Required locked inputs missing:\n'+'\n'.join(missing))
run10=json.loads(E10_1_RUN.read_text(encoding='utf-8'))
protocol=json.loads(E10_1_PROTOCOL.read_text(encoding='utf-8'))
artifact=joblib.load(MODEL_PATH)
expected_model_hash=run10.get('upstream_artifact_hashes',{}).get('model normal_vs_ad')
checks=pd.DataFrame([
 {'check':'P1-E10.1 passed','pass':str(run10.get('status','')).startswith('P1-E10.1 PASSED'),'detail':run10.get('status','')},
 {'check':'AHEPA declared READY','pass':run10.get('dataset_readiness',{}).get('AHEPA')=='READY','detail':run10.get('dataset_readiness',{})},
 {'check':'protocol hash authenticated','pass':sha256_file(E10_1_PROTOCOL)==run10.get('protocol_sha256'),'detail':sha256_file(E10_1_PROTOCOL)},
 {'check':'Normal-vs-AD is primary endpoint','pass':protocol.get('primary_endpoint')=='normal_vs_ad','detail':protocol.get('primary_endpoint')},
 {'check':'model hash authenticated','pass':sha256_file(MODEL_PATH)==expected_model_hash,'detail':sha256_file(MODEL_PATH)},
 {'check':'artifact task classes exact','pass':list(artifact.get('classes',[]))==['Normal','AD'],'detail':list(artifact.get('classes',[]))},
 {'check':'fitted estimator available','pass':hasattr(artifact.get('estimator'),'predict'),'detail':artifact.get('model_name')},
])
save_table(checks,'01_input_authentication_checks'); display(checks)
if not checks['pass'].all(): raise AssertionError('STOP — FROZEN INPUT AUTHENTICATION FAILED')
print('P1-E10.1 PROTOCOL AND CAUEEG MODEL AUTHENTICATED')

### 3. Lock the diagnosis-blind AHEPA preprocessing manifest

In [ ]:
manifest=pd.read_csv(AHEPA_MANIFEST,dtype={'participant_id':str})
eligible=manifest.loc[manifest.external_validation_eligible.astype(str).str.lower().isin(['true','1']) &
                      manifest.external_label.isin(['CN','AD'])].copy()
if eligible.participant_id.duplicated().any(): raise ValueError('Duplicate eligible AHEPA participant IDs')
if eligible.participant_id.nunique()!=65: warnings.warn(f'Expected 65 AHEPA CN/AD participants; found {eligible.participant_id.nunique()}')
blind=eligible[['participant_id','eeg_path']].copy()
label_lock=eligible[['participant_id','external_label']].copy()
label_lock['true_label']=label_lock.external_label.map({'CN':'Normal','AD':'AD'})
if label_lock.true_label.isna().any(): raise ValueError('Unexpected AHEPA label')
save_table(blind,'02_DIAGNOSIS_BLIND_AHEPA_PROCESSING_MANIFEST')
save_table(label_lock,'03_SEALED_AHEPA_LABEL_LOCK')
display(label_lock.groupby('true_label').participant_id.nunique().rename('participants'))
print('Feature extraction input columns:',blind.columns.tolist())

## Results

### 4. Apply frozen preprocessing and automatic epoch QA

In [ ]:
def norm_channel(value):
    text=str(value).strip().upper().replace('EEG','')
    text=re.sub(r'[-_ ]?(REF|LE|RE|AVG|AV)$','',text).strip(' -_')
    aliases={'FP1':'Fp1','FP2':'Fp2','F3':'F3','F4':'F4','C3':'C3','C4':'C4','P3':'P3','P4':'P4',
      'O1':'O1','O2':'O2','F7':'F7','F8':'F8','T3':'T3','T4':'T4','T5':'T5','T6':'T6',
      'T7':'T3','T8':'T4','P7':'T5','P8':'T6','FZ':'Fz','CZ':'Cz','PZ':'Pz'}
    return aliases.get(text,text)

def read_raw(path):
    path=Path(path); readers={'.set':mne.io.read_raw_eeglab,'.edf':mne.io.read_raw_edf,
      '.vhdr':mne.io.read_raw_brainvision,'.fif':mne.io.read_raw_fif}
    return readers[path.suffix.lower()](path,preload=False,verbose='ERROR')

def preprocess_continuous(data_uv,source_fs):
    data_uv=signal.detrend(np.asarray(data_uv,float),axis=1,type='linear')
    sos=signal.butter(FILTER_ORDER,BAND,btype='bandpass',fs=source_fs,output='sos')
    data_uv=signal.sosfiltfilt(sos,data_uv,axis=1)
    if not np.isclose(source_fs,FS):
        ratio=Fraction(FS/source_fs).limit_denominator(1000)
        data_uv=signal.resample_poly(data_uv,ratio.numerator,ratio.denominator,axis=1)
    return data_uv

def make_epochs(processed):
    trim=int(round(EDGE_TRIM_SECONDS*FS)); length=int(round(EPOCH_SECONDS*FS))
    usable=processed[:,trim:processed.shape[1]-trim]
    return np.asarray([usable[:,i:i+length] for i in range(0,usable.shape[1]-length+1,length)])

def epoch_metrics(epochs):
    finite=np.isfinite(epochs).all(axis=(1,2)); sd=np.std(epochs,axis=2)
    flat=(sd<=FLAT_SD_UV).sum(axis=1); ptp=np.ptp(epochs,axis=2).max(axis=1)
    rms=np.sqrt(np.mean(epochs**2,axis=(1,2))); step=np.max(np.abs(np.diff(epochs,axis=2)),axis=(1,2))
    extreme=np.mean(np.abs(epochs)>=EXTREME_UV,axis=(1,2))
    robust=np.maximum.reduce([np.abs(robust_z(np.log(ptp+1e-9))),np.abs(robust_z(np.log(rms+1e-9))),
                              np.abs(robust_z(np.log(step+1e-9))),np.abs(robust_z(extreme+1e-9))])
    accepted=(finite&(flat==0)&(ptp<PTP_LIMIT_UV)&(rms<RMS_LIMIT_UV)&
              (extreme<MAX_EXTREME_FRACTION)&(robust<ROBUST_SCORE_LIMIT))
    return pd.DataFrame({'finite':finite,'flat_channels':flat,'ptp_uv':ptp,'rms_uv':rms,
      'max_step_uv':step,'extreme_fraction':extreme,'robust_multimetric_score':robust,'accepted':accepted})

def evenly_distributed(indices,target):
    indices=np.asarray(indices,int)
    return indices if len(indices)<=target else indices[np.linspace(0,len(indices)-1,target).round().astype(int)]

epoch_audit=[]; participant_audit=[]
for number,row in enumerate(blind.itertuples(index=False),1):
    raw=None
    try:
        raw=read_raw(row.eeg_path); mapped=[norm_channel(x) for x in raw.ch_names]
        if len(set(mapped))!=len(mapped): raise ValueError('duplicate_channels_after_mapping')
        missing=[ch for ch in CHANNELS if ch not in mapped]
        if missing: raise ValueError(f'missing_locked_channels_{missing}')
        picks=[mapped.index(ch) for ch in CHANNELS]; source_fs=float(raw.info['sfreq'])
        data_uv=raw.get_data(picks=picks)*1e6
        candidate=make_epochs(preprocess_continuous(data_uv,source_fs))
        if not len(candidate): raise ValueError('no_candidate_epochs')
        qa=epoch_metrics(candidate); accepted=np.flatnonzero(qa.accepted.to_numpy())
        selected=evenly_distributed(accepted,TARGET_EPOCHS)
        for i,q in qa.iterrows():
            epoch_audit.append({'participant_id':row.participant_id,'epoch_index':i,
              **q.to_dict(),'selected':i in set(selected.tolist())})
        if len(selected)<MIN_EPOCHS: raise ValueError(f'only_{len(selected)}_accepted_epochs')
        kept=candidate[selected].astype('float32'); array_path=ARRAYS/f'{row.participant_id}_LOCKED_AHEPA_EPOCHS.npz'
        np.savez_compressed(array_path,epochs_uv=kept,fs_Hz=FS,channels=np.asarray(CHANNELS,dtype='U4'))
        participant_audit.append({'participant_id':row.participant_id,'source_fs_Hz':source_fs,
          'candidate_epochs':len(candidate),'accepted_epochs':len(accepted),'selected_epochs':len(selected),
          'processing_success':True,'array_path':str(array_path),'array_sha256':sha256_file(array_path),'error':''})
    except Exception as exc:
        participant_audit.append({'participant_id':row.participant_id,'source_fs_Hz':np.nan,
          'candidate_epochs':0,'accepted_epochs':0,'selected_epochs':0,'processing_success':False,
          'array_path':'','array_sha256':'','error':f'{type(exc).__name__}: {exc}'})
    finally:
        if raw is not None: raw.close()
    if number%10==0 or number==len(blind): print(f'[{number:02d}/{len(blind)}] AHEPA participants processed')

epoch_audit=pd.DataFrame(epoch_audit); participant_audit=pd.DataFrame(participant_audit)
save_table(epoch_audit,'04_AHEPA_AUTOMATED_EPOCH_QA_AUDIT')
save_table(participant_audit,'05_AHEPA_PARTICIPANT_PREPROCESSING_AUDIT')
display(participant_audit.processing_success.value_counts(dropna=False)); display(participant_audit.loc[~participant_audit.processing_success].head())

### 5. Extract the exact frozen P1-E08 feature representation

In [ ]:
def integrate_psd(psd,freqs,low,high):
    mask=(freqs>=low)&(freqs<high if high<40 else freqs<=high)
    return np.trapezoid(psd[...,mask],freqs[mask],axis=-1)
def epoch_features(epochs):
    freqs,psd=signal.welch(np.asarray(epochs,float),fs=FS,nperseg=int(4*FS),axis=-1)
    total=integrate_psd(psd,freqs,1,40)+1e-18; values=[]; names=[]; groups=[]
    for band,(low,high) in BANDS.items():
        power=integrate_psd(psd,freqs,low,high)
        for transform,prefix in [(np.log10(power+1e-18),f'logbp_{band}'),(power/total,f'relbp_{band}')]:
            values.append(transform); names += [f'{prefix}_{ch}' for ch in CHANNELS]; groups += ['spectral']*len(CHANNELS)
    use=(freqs>=1)&(freqs<=40); probability=psd[...,use]/(psd[...,use].sum(axis=-1,keepdims=True)+1e-18)
    entropy=-(probability*np.log(probability+1e-18)).sum(axis=-1)/np.log(probability.shape[-1])
    cumulative=np.cumsum(psd[...,use],axis=-1); edge=(cumulative>=.95*cumulative[...,-1][...,None]).argmax(axis=-1)
    sef95=freqs[use][edge]; alpha=(freqs>=8)&(freqs<=13); alpha_peak=freqs[alpha][np.argmax(psd[...,alpha],axis=-1)]
    for matrix,prefix in [(entropy,'spectral_entropy'),(sef95,'sef95_Hz'),(alpha_peak,'alpha_peak_Hz')]:
        values.append(matrix); names += [f'{prefix}_{ch}' for ch in CHANNELS]; groups += ['spectral']*len(CHANNELS)
    dx=np.diff(epochs,axis=-1); ddx=np.diff(dx,axis=-1); activity=np.var(epochs,axis=-1)
    mobility=np.sqrt(np.var(dx,axis=-1)/(activity+1e-18)); mobility_dx=np.sqrt(np.var(ddx,axis=-1)/(np.var(dx,axis=-1)+1e-18))
    complexity=mobility_dx/(mobility+1e-18); line_length=np.mean(np.abs(dx),axis=-1)
    for matrix,prefix in [(np.log10(activity+1e-18),'hjorth_log_activity'),(mobility,'hjorth_mobility'),
                          (complexity,'hjorth_complexity'),(line_length,'line_length')]:
        values.append(matrix); names += [f'{prefix}_{ch}' for ch in CHANNELS]; groups += ['temporal']*len(CHANNELS)
    return np.concatenate(values,axis=1),names,groups

feature_rows=[]; extraction=[]; aggregate_names=None
for row in participant_audit.loc[participant_audit.processing_success].itertuples(index=False):
    try:
        if sha256_file(row.array_path)!=row.array_sha256: raise ValueError('epoch_array_hash_mismatch')
        with np.load(row.array_path,allow_pickle=False) as p: epochs=np.asarray(p['epochs_uv'],float)
        matrix,names,groups=epoch_features(epochs)
        names_now=[f'median__{x}' for x in names]+[f'iqr__{x}' for x in names]
        if aggregate_names is None: aggregate_names=names_now
        elif names_now!=aggregate_names: raise ValueError('feature_schema_changed')
        vector=np.concatenate([np.median(matrix,axis=0),np.quantile(matrix,.75,axis=0)-np.quantile(matrix,.25,axis=0)])
        feature_rows.append({'participant_id':row.participant_id,'vector':vector})
        extraction.append({'participant_id':row.participant_id,'success':True,'features':len(vector),'error':''})
    except Exception as exc: extraction.append({'participant_id':row.participant_id,'success':False,'features':0,'error':repr(exc)})
extraction=pd.DataFrame(extraction); save_table(extraction,'06_AHEPA_FEATURE_EXTRACTION_AUDIT')
if not extraction.success.all(): raise RuntimeError('Feature extraction failed; do not predict.')
ids=[x['participant_id'] for x in feature_rows]; X_external=np.vstack([x['vector'] for x in feature_rows]).astype('float32')
columns=np.asarray(artifact['feature_indices'],int); stored_names=np.asarray(artifact['feature_names']).astype(str)
schema_ok=np.array_equal(np.asarray(aggregate_names)[columns],stored_names)
if not schema_ok: raise AssertionError('External feature schema does not match the locked model artifact.')
if not np.isfinite(X_external).all(): raise ValueError('Nonfinite external features')
np.savez_compressed(OUTPUT_ROOT/'AHEPA_LOCKED_NONSPIKING_FEATURE_MATRIX.npz',X=X_external,
                    participant_id=np.asarray(ids,dtype='U16'),feature_names=np.asarray(aggregate_names,dtype='U96'))
print('External feature matrix:',X_external.shape,'selected model features:',len(columns),'schema exact:',schema_ok)

### 6. Perform the single frozen participant-level evaluation

In [ ]:
evaluation=pd.DataFrame({'participant_id':ids}).merge(label_lock[['participant_id','true_label']],on='participant_id',validate='one_to_one')
estimator=artifact['estimator']; selected_X=X_external[:,columns]
prediction=estimator.predict(selected_X)
if not hasattr(estimator,'predict_proba'): raise RuntimeError('Locked Normal-vs-AD estimator lacks frozen probabilities')
classes=np.asarray(estimator.classes_).astype(str); ad_index=int(np.flatnonzero(classes=='AD')[0])
probability_ad=estimator.predict_proba(selected_X)[:,ad_index]
evaluation['predicted_label']=prediction; evaluation['probability_AD']=probability_ad
evaluation['correct']=evaluation.true_label.eq(evaluation.predicted_label)
save_table(evaluation,'07_LOCKED_AHEPA_PARTICIPANT_PREDICTIONS')

y_true=evaluation.true_label.to_numpy(); y_pred=evaluation.predicted_label.to_numpy(); y_bin=(y_true=='AD').astype(int)
tn,fp,fn,tp=confusion_matrix(y_true,y_pred,labels=['Normal','AD']).ravel()
metrics={'participants':len(evaluation),'normal':int((y_true=='Normal').sum()),'AD':int((y_true=='AD').sum()),
 'accuracy':accuracy_score(y_true,y_pred),'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
 'macro_f1':f1_score(y_true,y_pred,average='macro'),'mcc':matthews_corrcoef(y_true,y_pred),
 'sensitivity_AD':tp/(tp+fn),'specificity_Normal':tn/(tn+fp),
 'roc_auc':roc_auc_score(y_bin,probability_ad),'average_precision':average_precision_score(y_bin,probability_ad),
 'brier_score':brier_score_loss(y_bin,probability_ad)}

rng=np.random.default_rng(SEED); boot=[]
normal_idx=np.flatnonzero(y_true=='Normal'); ad_idx=np.flatnonzero(y_true=='AD')
for _ in range(BOOTSTRAP_REPS):
    idx=np.concatenate([rng.choice(normal_idx,len(normal_idx),replace=True),rng.choice(ad_idx,len(ad_idx),replace=True)])
    yt=y_true[idx]; yp=y_pred[idx]; yb=(yt=='AD').astype(int); pr=probability_ad[idx]
    cm=confusion_matrix(yt,yp,labels=['Normal','AD']); tni,fpi,fni,tpi=cm.ravel()
    boot.append({'balanced_accuracy':balanced_accuracy_score(yt,yp),'macro_f1':f1_score(yt,yp,average='macro'),
      'mcc':matthews_corrcoef(yt,yp),'sensitivity_AD':tpi/(tpi+fni),'specificity_Normal':tni/(tni+fpi),
      'roc_auc':roc_auc_score(yb,pr),'average_precision':average_precision_score(yb,pr),'brier_score':brier_score_loss(yb,pr)})
boot=pd.DataFrame(boot)
summary=[]
for name,value in metrics.items():
    if name in boot: summary.append({'metric':name,'estimate':value,'ci95_low':boot[name].quantile(.025),'ci95_high':boot[name].quantile(.975)})
    else: summary.append({'metric':name,'estimate':value,'ci95_low':np.nan,'ci95_high':np.nan})
summary=pd.DataFrame(summary); report=pd.DataFrame(classification_report(y_true,y_pred,labels=['Normal','AD'],output_dict=True,zero_division=0)).T.reset_index(names='label')
save_table(summary,'08_AHEPA_EXTERNAL_VALIDATION_METRICS'); save_table(report,'09_AHEPA_CLASSIFICATION_REPORT')
display(summary); display(report)

### 7. Publication-ready external-validation figures

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,3.6))
matrix=confusion_matrix(evaluation.true_label,evaluation.predicted_label,labels=['Normal','AD'])
sns.heatmap(matrix,annot=True,fmt='d',cmap='Blues',cbar=False,ax=axes[0],xticklabels=['Normal','AD'],yticklabels=['Normal','AD'])
axes[0].set(title='AHEPA confusion matrix',xlabel='Predicted',ylabel='True')
sns.boxplot(data=evaluation,x='true_label',y='probability_AD',hue='true_label',legend=False,ax=axes[1],order=['Normal','AD'])
axes[1].set(title='Frozen AD probability',xlabel='True group',ylabel='Probability of AD')
fpr,tpr,_=roc_curve((evaluation.true_label=='AD').astype(int),evaluation.probability_AD)
axes[2].plot(fpr,tpr,color='#0072B2',label=f"AUC = {metrics['roc_auc']:.3f}"); axes[2].plot([0,1],[0,1],'--',color='grey')
axes[2].set(title='Participant-level ROC',xlabel='False-positive rate',ylabel='True-positive rate'); axes[2].legend(frameon=False)
savefig(fig,'Figure_01_AHEPA_frozen_external_validation',12,3.6); plt.show()

fig,ax=plt.subplots(); order=evaluation.sort_values('probability_AD').reset_index(drop=True)
colors=order.true_label.map({'Normal':'#4C78A8','AD':'#E45756'})
ax.scatter(np.arange(len(order)),order.probability_AD,c=colors,s=20)
ax.set(title='Frozen CAUEEG model probabilities across AHEPA participants',xlabel='Participants sorted by AD probability',ylabel='Probability of AD')
savefig(fig,'Figure_02_AHEPA_participant_probability_distribution',8,4); plt.show()

## Checks

### 8. Lock the external result and provenance

In [ ]:
retention=participant_audit.processing_success.mean()
final_checks=pd.DataFrame([
 {'check':'all upstream authentication gates passed','pass':checks['pass'].all(),'detail':'P1-E10.1 and P1-E08.1 artifact'},
 {'check':'preprocessing remained diagnosis-blind','pass':set(blind.columns)=={'participant_id','eeg_path'},'detail':blind.columns.tolist()},
 {'check':'FTD excluded from confirmatory endpoint','pass':not label_lock.true_label.eq('FTD').any(),'detail':label_lock.true_label.value_counts().to_dict()},
 {'check':'participant IDs unique','pass':not evaluation.participant_id.duplicated().any(),'detail':len(evaluation)},
 {'check':'minimum participant retention reached','pass':retention>=MIN_PARTICIPANT_RETENTION,'detail':retention},
 {'check':'minimum six accepted epochs per evaluated participant','pass':participant_audit.loc[participant_audit.processing_success].selected_epochs.ge(MIN_EPOCHS).all(),'detail':participant_audit.loc[participant_audit.processing_success].selected_epochs.min()},
 {'check':'external feature schema exactly matched','pass':schema_ok,'detail':len(aggregate_names)},
 {'check':'frozen feature subset used','pass':selected_X.shape[1]==len(artifact['feature_indices']),'detail':selected_X.shape},
 {'check':'one prediction per retained participant','pass':len(evaluation)==participant_audit.processing_success.sum(),'detail':len(evaluation)},
 {'check':'both endpoint classes retained','pass':set(evaluation.true_label)=={'Normal','AD'},'detail':evaluation.true_label.value_counts().to_dict()},
 {'check':'no model fitting performed','pass':True,'detail':'loaded fitted estimator; predict/predict_proba only'},
 {'check':'no tuning, recalibration or threshold selection','pass':True,'detail':'artifact prediction rule unchanged'},
 {'check':'bootstrap is participant-level stratified','pass':len(boot)==BOOTSTRAP_REPS,'detail':len(boot)},
])
save_table(final_checks,'10_final_integrity_checks'); display(final_checks)
status=('P1-E10.2 PASSED — AHEPA FROZEN NORMAL-vs-AD EXTERNAL VALIDATION LOCKED'
        if final_checks['pass'].all() else 'STOP — P1-E10.2 CHECKS FAILED; DO NOT INTERPRET EXTERNAL PERFORMANCE')
pred_path=TABLES/'07_LOCKED_AHEPA_PARTICIPANT_PREDICTIONS.csv'; metrics_path=TABLES/'08_AHEPA_EXTERNAL_VALIDATION_METRICS.csv'
run_record={'experiment':'P1-E10.2 frozen CAUEEG Normal-vs-AD external validation on AHEPA',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'status':status,'source_p1_e10_1_run_sha256':sha256_file(E10_1_RUN),
 'source_protocol_sha256':sha256_file(E10_1_PROTOCOL),'source_model_sha256':sha256_file(MODEL_PATH),
 'eligible_input_participants':len(blind),'evaluated_participants':len(evaluation),'retention_fraction':retention,
 'endpoint':'Normal vs AD','external_dataset':'AHEPA ds004504','model_refit':False,
 'threshold_changed':False,'calibration_changed':False,'predictions_csv':str(pred_path),
 'predictions_sha256':sha256_file(pred_path),'metrics_csv':str(metrics_path),'metrics_sha256':sha256_file(metrics_path),
 'metrics':{k:float(v) if isinstance(v,(float,np.floating)) else int(v) for k,v in metrics.items()},
 'python':sys.version,'platform':platform.platform()}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not final_checks['pass'].all(): raise AssertionError(status)

## Next Steps

After a P1-E10.2 PASS:

1. Interpret performance as external participant-level transportability, not as new model development.
2. Report all exclusions and the complete participant-level confidence intervals.
3. Do not change the CAUEEG model in response to AHEPA performance.
4. Proceed to **P1-E10.3**, the separately frozen Dryad validation.
